# BART + LoRA Training Run

Official fine-tuning run for the milestone. Trains on a fixed 30k-row subset
of the training data (out of ~317k available) -- a deliberate time/quality
tradeoff to keep training to roughly 30-45 min on a Colab T4, rather than
the ~7 hours a full-dataset run would take. See `docs/model_comparison.md`
for the tradeoff discussion.

**Assumes `notebooks/00_colab_setup.ipynb` has already been run this
session** (repo cloned, dependencies installed, API keys loaded, GPU
confirmed, data downloaded/restored and preprocessed).

For hyperparameter exploration, alternate subset sizes, or other ad hoc
experiments, use a notebook under `experiments/` instead -- keep this
notebook clean and reproducible.

In [ ]:
# 1. Confirm environment is ready (run 00_colab_setup.ipynb first if this fails)
import os

REPO_DIR = '/content/Review-Summarization-LLM'
assert os.path.exists(REPO_DIR), "Repo not found -- run 00_colab_setup.ipynb first."
%cd {REPO_DIR}
assert os.path.exists('data/processed/train.csv'), "Processed data not found -- run 00_colab_setup.ipynb first."
print('Environment ready.')

In [ ]:
# 2. Create a fixed, reproducible 30k-row training subset (+ proportional val subset)
import pandas as pd

TRAIN_SUBSET_SIZE = 30000
VAL_SUBSET_SIZE = 3000
SEED = 42

train_df = pd.read_csv('data/processed/train.csv', engine='python', on_bad_lines='warn')
val_df = pd.read_csv('data/processed/val.csv', engine='python', on_bad_lines='warn')

train_subset = train_df.sample(n=min(TRAIN_SUBSET_SIZE, len(train_df)), random_state=SEED)
val_subset = val_df.sample(n=min(VAL_SUBSET_SIZE, len(val_df)), random_state=SEED)

train_subset.to_csv('data/processed/train_subset_30k.csv', index=False)
val_subset.to_csv('data/processed/val_subset_30k.csv', index=False)

print(f'Train subset: {len(train_subset)} rows')
print(f'Val subset: {len(val_subset)} rows')

In [ ]:
# 3. Run training on the subset (~30-45 min on a T4)
import time
from training.train_bart import train

start = time.time()
train(
    train_path='data/processed/train_subset_30k.csv',
    val_path='data/processed/val_subset_30k.csv',
    output_dir='outputs/bart_lora_30k',
)
elapsed = time.time() - start
print(f'Training completed in {elapsed/60:.1f} minutes')

In [ ]:
# 4. Back up the trained adapter to Drive immediately (checkpoints are gitignored --
# too large for GitHub -- so Drive is the only persistent copy across sessions)
BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
!mkdir -p {BACKUP_DIR}/bart_lora_30k
!cp -r outputs/bart_lora_30k/final/* {BACKUP_DIR}/bart_lora_30k/
print('Checkpoint backed up to Drive.')

## Sanity check: generate on a few held-out validation examples

Quick qualitative check that fine-tuning actually improved on the zero-shot
BART baseline seen in `00_colab_setup.ipynb`'s smoke test -- not a
substitute for the full ROUGE/BERTScore evaluation (see Next Steps below).

In [ ]:
# 5. Load the fine-tuned adapter and compare its output to the reference Summary
from transformers import BartTokenizer, BartForConditionalGeneration
from peft import PeftModel
import pandas as pd

base_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
tokenizer = BartTokenizer.from_pretrained('outputs/bart_lora_30k/final')
model = PeftModel.from_pretrained(base_model, 'outputs/bart_lora_30k/final')
model.eval()

sample = pd.read_csv('data/processed/val_subset_30k.csv', engine='python', on_bad_lines='warn').sample(5, random_state=1)

for _, row in sample.iterrows():
    inputs = tokenizer(row['Text'], return_tensors='pt', max_length=1024, truncation=True)
    output_ids = model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True)
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print('Review:', row['Text'][:150], '...')
    print('Reference Summary:', row['Summary'])
    print('Fine-tuned BART Summary:', generated)
    print('---')

## Next steps

Run the full reference-based evaluation on this checkpoint's predictions
across the val/test set using `evaluation/evaluate.py reference` mode
(ROUGE-1/2/L + BERTScore against the real `Summary` column), then update
`docs/evaluation_report.md` with the results.